In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
import torchvision
import torchvision.transforms as transforms
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import time
import warnings
warnings.filterwarnings('ignore')

# Configuration
MAX_RUNTIME_HOURS = 11.5
MAX_RUNTIME_SECONDS = MAX_RUNTIME_HOURS * 3600
EPOCHS = 25
BATCH_SIZE = 128
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f"Using device: {DEVICE}")
print(f"Maximum runtime: {MAX_RUNTIME_HOURS} hours")

Path("results_resnet").mkdir(exist_ok=True)

# ==================== DATASET ====================
def get_data_loaders():
    transform_train = transforms.Compose([
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
    ])

    transform_test = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
    ])

    trainset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform_train)
    testset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform_test)

    trainloader = DataLoader(trainset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
    testloader = DataLoader(testset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
    
    return trainloader, testloader

# ==================== MODULAR RESNET-18 ====================
class BasicBlock(nn.Module):
    expansion = 1

    def __init__(self, in_planes, planes, stride=1, use_bn=True, use_res=True):
        super(BasicBlock, self).__init__()
        self.use_res = use_res
        self.use_bn = use_bn
        
        self.conv1 = nn.Conv2d(in_planes, planes, kernel_size=3, stride=stride, padding=1, bias=not use_bn)
        self.bn1 = nn.BatchNorm2d(planes) if use_bn else nn.Identity()
        self.conv2 = nn.Conv2d(planes, planes, kernel_size=3, stride=1, padding=1, bias=not use_bn)
        self.bn2 = nn.BatchNorm2d(planes) if use_bn else nn.Identity()

        self.shortcut = nn.Sequential()
        if stride != 1 or in_planes != self.expansion * planes:
            # For shortcut, we also need to respect the use_bn flag
            layers = [nn.Conv2d(in_planes, self.expansion * planes, kernel_size=1, stride=stride, bias=not use_bn)]
            if use_bn:
                layers.append(nn.BatchNorm2d(self.expansion * planes))
            self.shortcut = nn.Sequential(*layers)

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        
        if self.use_res:
            out += self.shortcut(x)
            
        out = F.relu(out)
        return out

class ResNet18_Custom(nn.Module):
    def __init__(self, num_classes=10, use_bn=True, use_res=True):
        super(ResNet18_Custom, self).__init__()
        self.in_planes = 64
        self.use_bn = use_bn
        self.use_res = use_res

        # CIFAR-10 optimized stem (3x3 conv instead of 7x7)
        self.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=not use_bn)
        self.bn1 = nn.BatchNorm2d(64) if use_bn else nn.Identity()
        
        # Layers
        self.layer1 = self._make_layer(64, 2, stride=1)
        self.layer2 = self._make_layer(128, 2, stride=2)
        self.layer3 = self._make_layer(256, 2, stride=2)
        self.layer4 = self._make_layer(512, 2, stride=2)
        
        self.linear = nn.Linear(512 * BasicBlock.expansion, num_classes)
        
        # Initialization
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, (nn.BatchNorm2d, nn.GroupNorm)):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)

    def _make_layer(self, planes, num_blocks, stride):
        strides = [stride] + [1]*(num_blocks-1)
        layers = []
        for stride in strides:
            layers.append(BasicBlock(self.in_planes, planes, stride, self.use_bn, self.use_res))
            self.in_planes = planes * BasicBlock.expansion
        return nn.Sequential(*layers)

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.layer1(out)
        out = self.layer2(out)
        out = self.layer3(out)
        out = self.layer4(out)
        out = F.avg_pool2d(out, 4)
        out = out.view(out.size(0), -1)
        out = self.linear(out)
        return out

# ==================== METRICS LOGGER ====================
class MetricsLoggerCNN:
    def __init__(self, model, model_name):
        self.model = model
        self.model_name = model_name
        self.metrics = []
        self.activations = {}
        self.hooks = []
        
        # Register hooks on the output of each ResNet Layer Group
        layers = [model.layer1, model.layer2, model.layer3, model.layer4]
        for idx, layer in enumerate(layers):
            hook = layer.register_forward_hook(self._make_hook(f'block_{idx+1}'))
            self.hooks.append(hook)

    def _make_hook(self, name):
        def hook(module, input, output):
            # Output is (B, C, H, W). We AvgPool to (B, C) to check representation rank
            pooled = F.adaptive_avg_pool2d(output, (1, 1)).flatten(1)
            self.activations[name] = pooled.detach()
        return hook

    def compute_stable_rank(self, x):
        try:
            if x.abs().max() < 1e-8: return 1.0
            s = torch.linalg.svdvals(x.float())
            s = s[s > 1e-8]
            if len(s) == 0: return 1.0
            stable_rank = ((s ** 2).sum() / (s[0] ** 2)).item()
            return max(1.0, stable_rank)
        except:
            return 1.0

    def compute_active_neurons(self, x):
        try:
            return (x > 0).float().mean().item()
        except:
            return 0.0

    def log_metrics(self, epoch, train_loss, val_loss, val_acc):
        try:
            metrics_dict = {
                'model': self.model_name,
                'epoch': epoch,
                'train_loss': train_loss,
                'val_loss': val_loss,
                'val_acc': val_acc
            }
            
            for name, activation in self.activations.items():
                metrics_dict[f'rank_{name}'] = self.compute_stable_rank(activation)
                metrics_dict[f'active_{name}'] = self.compute_active_neurons(activation)
            
            self.metrics.append(metrics_dict)
            self.activations.clear()
            return metrics_dict
        except Exception as e:
            print(f"Log Error: {e}")
            return None

    def get_dataframe(self):
        return pd.DataFrame(self.metrics)
    
    def cleanup(self):
        for h in self.hooks: h.remove()

# ==================== PLOTTING ====================
def save_resnet_plots(all_metrics):
    try:
        print("\n📊 Generating ResNet plots...")
        
        # Plot A: Rank Evolution (Line Plot)
        fig, ax = plt.subplots(figsize=(10, 6))
        
        for model_name, df in all_metrics.items():
            if len(df) == 0: continue
            
            # Get the rank of the DEEPEST layer (block_4)
            if 'rank_block_4' in df.columns:
                ax.plot(df['epoch'], df['rank_block_4'], marker='.', label=f"{model_name} (Deepest Layer)", linewidth=2)
        
        ax.set_xlabel('Epoch')
        ax.set_ylabel('Stable Rank (Block 4)')
        ax.set_title('ResNet-18: Rank Collapse in Deepest Layer')
        ax.legend()
        ax.grid(True, alpha=0.3)
        plt.savefig('results_resnet/plot_resnet_rank.png', dpi=150, bbox_inches='tight')
        plt.close()
        
        # Plot B: Accuracy
        fig, ax = plt.subplots(figsize=(10, 6))
        for model_name, df in all_metrics.items():
            if len(df) > 0:
                ax.plot(df['epoch'], df['val_acc'], marker='.', label=model_name)
        
        ax.set_title('ResNet-18: Accuracy Impact')
        ax.set_ylabel('Validation Accuracy')
        ax.axhline(10, color='gray', linestyle='--')
        ax.legend()
        plt.savefig('results_resnet/plot_resnet_acc.png', dpi=150, bbox_inches='tight')
        plt.close()
        
    except Exception as e:
        print(f"Plot Error: {e}")

# ==================== MAIN ====================
def run_resnet_experiment():
    start_time = time.time()
    trainloader, testloader = get_data_loaders()
    
    # 4 Configs to prove universality
    configs = {
        'ResNet_Star':   {'bn': True,  'res': True},
        'ResNet_NoBN':   {'bn': False, 'res': True},
        'ResNet_NoRes':  {'bn': True,  'res': False},
        'ResNet_Naked':  {'bn': False, 'res': False},
    }
    
    all_metrics = {}
    
    for model_name, cfg in configs.items():
        print(f"\n🚀 Training {model_name}...")
        
        try:
            model = ResNet18_Custom(use_bn=cfg['bn'], use_res=cfg['res']).to(DEVICE)
            optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
            criterion = nn.CrossEntropyLoss()
            logger = MetricsLoggerCNN(model, model_name)
            
            for epoch in range(1, EPOCHS + 1):
                # Check Time
                if time.time() - start_time > MAX_RUNTIME_SECONDS: break
                
                # Train (Simplified loop)
                model.train()
                t_loss, batches = 0, 0
                for i, (d, t) in enumerate(trainloader):
                    d, t = d.to(DEVICE), t.to(DEVICE)
                    optimizer.zero_grad()
                    out = model(d)
                    loss = criterion(out, t)
                    loss.backward()
                    optimizer.step()
                    t_loss += loss.item()
                    batches += 1
                    if i >= 5: break # Fast metrics on first 5 batches
                
                # Validate
                model.eval()
                correct, total = 0, 0
                with torch.no_grad():
                    for d, t in testloader:
                        d, t = d.to(DEVICE), t.to(DEVICE)
                        out = model(d)
                        _, p = out.max(1)
                        total += t.size(0)
                        correct += p.eq(t).sum().item()
                val_acc = 100.*correct/total
                
                logger.log_metrics(epoch, t_loss/batches, 0.0, val_acc)
                print(f"  Ep {epoch}: Acc {val_acc:.2f}%")
            
            all_metrics[model_name] = logger.get_dataframe()
            all_metrics[model_name].to_csv(f'results_resnet/{model_name}.csv', index=False)
            logger.cleanup()
            del model
            
        except Exception as e:
            print(f"  Failed {model_name}: {e}")
            
    save_resnet_plots(all_metrics)
    print("Done!")

if __name__ == "__main__":
    run_resnet_experiment()

Using device: cuda
Maximum runtime: 11.5 hours


100%|██████████| 170M/170M [00:02<00:00, 58.9MB/s]



🚀 Training ResNet_Star...
  Ep 1: Acc 18.77%
  Ep 2: Acc 16.64%
  Ep 3: Acc 21.65%
  Ep 4: Acc 25.27%
  Ep 5: Acc 24.62%
  Ep 6: Acc 31.25%
  Ep 7: Acc 36.55%
  Ep 8: Acc 34.45%
  Ep 9: Acc 37.18%
  Ep 10: Acc 40.81%
  Ep 11: Acc 33.01%
  Ep 12: Acc 40.26%
  Ep 13: Acc 40.56%
  Ep 14: Acc 38.61%
  Ep 15: Acc 39.34%
  Ep 16: Acc 42.24%
  Ep 17: Acc 41.91%
  Ep 18: Acc 40.49%
  Ep 19: Acc 47.38%
  Ep 20: Acc 45.37%
  Ep 21: Acc 45.56%
  Ep 22: Acc 44.61%
  Ep 23: Acc 40.90%
  Ep 24: Acc 44.11%
  Ep 25: Acc 47.69%

🚀 Training ResNet_NoBN...
  Ep 1: Acc 10.00%
  Ep 2: Acc 10.00%
  Ep 3: Acc 16.11%
  Ep 4: Acc 16.04%
  Ep 5: Acc 18.10%
  Ep 6: Acc 17.25%
  Ep 7: Acc 13.92%
  Ep 8: Acc 17.74%
  Ep 9: Acc 17.93%
  Ep 10: Acc 20.16%
  Ep 11: Acc 23.40%
  Ep 12: Acc 23.43%
  Ep 13: Acc 23.82%
  Ep 14: Acc 22.97%
  Ep 15: Acc 22.40%
  Ep 16: Acc 27.81%
  Ep 17: Acc 26.05%
  Ep 18: Acc 21.46%
  Ep 19: Acc 28.02%
  Ep 20: Acc 28.55%
  Ep 21: Acc 27.17%
  Ep 22: Acc 31.30%
  Ep 23: Acc 30.84%
  Ep